Step 1: Exploring the data

In [2]:
import pandas as pd

In [3]:
df=pd.read_csv('dataset/archive (5)/Resume/Resume.csv')

In [121]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2481 entries, 0 to 2480
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   ID            2481 non-null   int64 
 1   Resume_str    2481 non-null   str   
 2   Resume_html   2481 non-null   str   
 3   Category      2481 non-null   str   
 4   Resume_clean  2481 non-null   str   
 5   tokens        2481 non-null   object
dtypes: int64(1), object(1), str(4)
memory usage: 66.3+ MB


In [5]:
df.head(3)

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [6]:
df.shape

(2484, 4)

In [7]:
df['Category'].unique()

<ArrowStringArray>
[                    'HR',               'DESIGNER', 'INFORMATION-TECHNOLOGY',
                'TEACHER',               'ADVOCATE',   'BUSINESS-DEVELOPMENT',
             'HEALTHCARE',                'FITNESS',            'AGRICULTURE',
                    'BPO',                  'SALES',             'CONSULTANT',
          'DIGITAL-MEDIA',             'AUTOMOBILE',                   'CHEF',
                'FINANCE',                'APPAREL',            'ENGINEERING',
             'ACCOUNTANT',           'CONSTRUCTION',       'PUBLIC-RELATIONS',
                'BANKING',                   'ARTS',               'AVIATION']
Length: 24, dtype: str

In [8]:
df['Category'].value_counts()

Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ADVOCATE                  118
CHEF                      118
FINANCE                   118
ENGINEERING               118
ACCOUNTANT                118
FITNESS                   117
AVIATION                  117
SALES                     116
HEALTHCARE                115
CONSULTANT                115
BANKING                   115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22
Name: count, dtype: int64

In [9]:
df.duplicated().sum()

np.int64(0)

In [10]:
df['Resume_str'].duplicated().sum()

np.int64(2)

In [11]:
df[df['Resume_str'].duplicated(keep=False)]


,ID,Resume_str,Resume_html,Category
1490,19147603,FINANCE OFFICER Professional ...,"<div class=""fontsize fontface vmargins hmargin...",FINANCE
1509,28398216,FINANCE OFFICER Professional ...,"<div class=""fontsize fontface vmargins hmargin...",FINANCE
2444,16850314,STOREKEEPER II Professional Sum...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION
2483,37473139,STOREKEEPER II Professional Sum...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION


In [12]:
df = df.drop_duplicates(subset='Resume_str', keep='first').reset_index(drop=True)


In [13]:
df.shape

(2482, 4)

In [14]:
df['Resume_str'][0]

"         HR ADMINISTRATOR/MARKETING ASSOCIATE\n\nHR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management.   Respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to customer service.         Highlights         Focused on customer satisfaction  Team management  Marketing savvy  Conflict resolution techniques     Training and development  Skilled multi-tasker  Client relations specialist           Accomplishments      Missouri DOT Supervisor Training Certification  Certified by IHG in Customer Loyalty and Marketing by Segment   Hilton Worldwide General Manager Training Certification  Accomplished Trainer for cross server hospitality systems such as    Hilton OnQ  ,   Micros    Opera PMS   , Fidelio    OPERA    Reservation System (ORS) ,   Holidex    Completed courses and seminars in customer service, sales strategies, inventory control, loss pr

In [15]:
print(repr(df['Resume_str'].iloc[1200]))


"         SENIOR CONSULTANT           Experience      Senior Consultant  ,     09/2015   to   Current     Company Name   –   City  ,   State      Manage the relationship between CVS Health Med D enrollment operations and EGS (Expert Global Solutions), a.  vendor contracted to process member centric requests and operational processes with 230+ employees.  Engage.  with site directors, operations managers, HR, trainers, workforce consultants, and supervisors to strategically.  resolve workflow and deliverable issues.  Ensure continued service delivery and quality satisfaction from EGS and a successful working relationship between CVS and EGS.  Travel to two main sites bi-monthly during Med D's annual enrollment period to ensure successful training execution.  Set service expectations for each line of business.  Successfully brought up a vendor site with 100+ employees with a 2-month period, including access to all systems, training and escalations.  Raised quality from 70% to an average 

Step 2 - A: cleaning 

In [16]:
import re

In [17]:
def clean_text(text: str)->str:
    #remove null/control bytes
    text=re.sub(r"[\x00-\x08\x0b\x0c\x0e\x0f]", "", text)

    #normalize dashes
    text=re.sub(r"[‐-‒–—―−\uff0d]+", "-",text)

    # Collapse repeated spaces/tabs, without removing newlines
    text = re.sub(r"[^\S\r\n]+", " ", text)
    #collapse multiple newlines and removing spaces around them
    text = re.sub(r"[ \t\r]*\n[ \t\r\n]*", "\n", text)

    return text.strip()


In [18]:
df['Resume_clean']=df['Resume_str'].apply(clean_text)

In [19]:
len(df['Resume_clean'])

2482

In [20]:
count=0
for i in range(len(df['Resume_clean'])):
    if len(df['Resume_clean'][i])<20:
        count+=1
        print(i)
    else:
        continue
print("count of junk or empty resumes: ",count)


656
count of junk or empty resumes:  1


In [21]:
df = df.drop(index=656).reset_index(drop=True)


#step 2 - B: Tokenization 

In [22]:
import spacy

In [23]:
nlp = spacy.load("en_core_web_md")

In [24]:
import string

def preprocessing(text: str):
    doc = nlp(text, disable=["parser", "ner"])
    filtered = []
    for token in doc:
        if token.is_stop or token.is_punct or token.is_space:
            continue
        lemma = token.lemma_.lower().strip(string.punctuation)
        if len(lemma) < 2 or not any(c.isalpha() for c in lemma):
            continue
        filtered.append(lemma)
    return filtered

preprocessing(df['Resume_clean'][1200])

['managing',
 'consultant',
 'summary',
 'highly',
 'accomplished',
 'management',
 'consultant',
 'senior',
 'business',
 'analyst',
 'verifiable',
 'track',
 'record',
 'manage',
 'complex',
 'strategy',
 'project',
 'exceed',
 'client',
 'expectation',
 'demonstrate',
 'skill',
 'business',
 'process',
 'management',
 'process',
 'redesign',
 'specialize',
 'end',
 'end',
 'business',
 'process',
 'management',
 'lifecycle',
 'extensive',
 'experience',
 'integration',
 'implementation',
 'organizational',
 'transformational',
 'effort',
 'public',
 'financial',
 'services',
 'sectors',
 'designing',
 'process',
 'system',
 'improvement',
 'increase',
 'productivity',
 'reduce',
 'cost',
 'strong',
 'interpersonal',
 'skill',
 'highly',
 'adept',
 'manage',
 'broad',
 'stakeholder',
 'community',
 'support',
 'development',
 'cohesive',
 'strategic',
 'vision',
 'disparate',
 'group',
 'skills',
 'business',
 'process',
 'improvement',
 'redesign',
 'agile',
 'scrum',
 'sdlc',
 'bus

In [25]:
df['tokens']=df['Resume_clean'].apply(preprocessing)

In [26]:
df['tokens'].apply(len).describe()

count    2481.00000
mean      573.49738
std       258.00497
min        67.00000
25%       464.00000
50%       536.00000
75%       661.00000
max      3473.00000
Name: tokens, dtype: float64

In [27]:
count=0
check=[]
for i in range(len(df['tokens'])):
    if len(df['tokens'][i])<100:
        count+=1
        print(i)
        check.append(df['Category'][i])
    else:
        continue
print("count of junk or empty resumes: ",count)
check

130
153
1038
1048
1101
1929
count of junk or empty resumes:  6


['DESIGNER', 'DESIGNER', 'SALES', 'SALES', 'SALES', 'CONSTRUCTION']

In [28]:
df['tokens'][153]

['floral',
 'designer',
 'skills',
 'billings',
 'cash',
 'handling',
 'cashier',
 'creativity',
 'customer',
 'service',
 'magic',
 'pick',
 'pos',
 'experience',
 'jan',
 'current',
 'company',
 'city',
 'state',
 'floral',
 'designer',
 'jan',
 'company',
 'city',
 'state',
 'designer',
 'jan',
 'company',
 'city',
 'state',
 'assign',
 'errand',
 'duty',
 'customer',
 'service',
 'design',
 'work',
 'event',
 'set',
 'magic',
 'city',
 'floral',
 'billings',
 'mt',
 'customer',
 'service',
 'miscellaneous',
 'assign',
 'duty',
 'floral',
 'designer',
 'delivery',
 'driver',
 'jan',
 'jan',
 'company',
 'city',
 'state',
 'assign',
 'duty',
 'education',
 'training',
 'work',
 'floral',
 'design',
 'certificate',
 'fall',
 'range',
 'community',
 'college',
 'range',
 'community',
 'college',
 'work',
 'floral',
 'design',
 'certificate',
 'spring',
 'associates',
 'horticulture',
 'fall',
 'range',
 'community',
 'college',
 'horticulture',
 'spring',
 'colorado',
 'state',
 'unive

Step 3: Building the model 

In [29]:
from sklearn.ensemble import RandomForestClassifier
from  sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression

In [30]:
from sklearn.model_selection import train_test_split

In [31]:
x_train,x_test,y_train,y_test=train_test_split(df['tokens'],df['Category'],test_size=0.2,stratify=df['Category'],random_state=42)

In [32]:
from sklearn.feature_extraction.text import TfidfVectorizer

# your tokens are already cleaned/lemmatized, so join them back into strings
x_train_str = x_train.apply(' '.join)
x_test_str = x_test.apply(' '.join)

vectorizer = TfidfVectorizer()
x_train_vec = vectorizer.fit_transform(x_train_str)
x_test_vec = vectorizer.transform(x_test_str)


Baseline models

In [33]:
model_1=LogisticRegression(class_weight='balanced')
model_1.fit(x_train_vec,y_train)
y_pred_1=model_1.predict(x_test_vec)
print(classification_report(y_test,y_pred_1))

                        precision    recall  f1-score   support

            ACCOUNTANT       0.57      0.83      0.68        24
              ADVOCATE       0.42      0.42      0.42        24
           AGRICULTURE       0.70      0.54      0.61        13
               APPAREL       0.60      0.16      0.25        19
                  ARTS       0.46      0.29      0.35        21
            AUTOMOBILE       0.57      0.57      0.57         7
              AVIATION       0.86      0.78      0.82        23
               BANKING       0.88      0.65      0.75        23
                   BPO       0.67      0.50      0.57         4
  BUSINESS-DEVELOPMENT       0.49      0.79      0.60        24
                  CHEF       0.81      0.71      0.76        24
          CONSTRUCTION       0.81      0.77      0.79        22
            CONSULTANT       0.43      0.13      0.20        23
              DESIGNER       0.81      0.81      0.81        21
         DIGITAL-MEDIA       0.79      

In [34]:
model_2=RandomForestClassifier(class_weight='balanced', random_state=42)
model_2.fit(x_train_vec,y_train)
y_pred_2=model_2.predict(x_test_vec)
print(classification_report(y_test,y_pred_2))

                        precision    recall  f1-score   support

            ACCOUNTANT       0.54      0.92      0.68        24
              ADVOCATE       0.78      0.58      0.67        24
           AGRICULTURE       1.00      0.54      0.70        13
               APPAREL       0.86      0.32      0.46        19
                  ARTS       0.50      0.10      0.16        21
            AUTOMOBILE       1.00      0.14      0.25         7
              AVIATION       0.77      0.87      0.82        23
               BANKING       0.86      0.52      0.65        23
                   BPO       0.00      0.00      0.00         4
  BUSINESS-DEVELOPMENT       0.49      0.71      0.58        24
                  CHEF       0.82      0.75      0.78        24
          CONSTRUCTION       0.75      0.82      0.78        22
            CONSULTANT       0.67      0.26      0.38        23
              DESIGNER       0.78      0.86      0.82        21
         DIGITAL-MEDIA       0.69      

In [35]:
model_3=KNeighborsClassifier()
model_3.fit(x_train_vec,y_train)
y_pred_3=model_3.predict(x_test_vec)
print(classification_report(y_test,y_pred_3))

                        precision    recall  f1-score   support

            ACCOUNTANT       0.51      0.88      0.65        24
              ADVOCATE       0.27      0.58      0.37        24
           AGRICULTURE       0.57      0.31      0.40        13
               APPAREL       0.33      0.21      0.26        19
                  ARTS       0.46      0.29      0.35        21
            AUTOMOBILE       0.25      0.29      0.27         7
              AVIATION       0.69      0.39      0.50        23
               BANKING       0.63      0.52      0.57        23
                   BPO       0.00      0.00      0.00         4
  BUSINESS-DEVELOPMENT       0.46      0.75      0.57        24
                  CHEF       0.86      0.75      0.80        24
          CONSTRUCTION       0.61      0.77      0.68        22
            CONSULTANT       0.36      0.22      0.27        23
              DESIGNER       0.71      0.71      0.71        21
         DIGITAL-MEDIA       0.65      

In [36]:
from sklearn.svm import LinearSVC

model_svm = LinearSVC(class_weight='balanced')
model_svm.fit(x_train_vec, y_train)
y_pred_svm = model_svm.predict(x_test_vec)
print(classification_report(y_test, y_pred_svm, zero_division=0))


                        precision    recall  f1-score   support

            ACCOUNTANT       0.57      0.83      0.68        24
              ADVOCATE       0.70      0.67      0.68        24
           AGRICULTURE       0.89      0.62      0.73        13
               APPAREL       0.54      0.37      0.44        19
                  ARTS       0.50      0.38      0.43        21
            AUTOMOBILE       0.75      0.43      0.55         7
              AVIATION       0.86      0.78      0.82        23
               BANKING       0.81      0.74      0.77        23
                   BPO       0.67      0.50      0.57         4
  BUSINESS-DEVELOPMENT       0.54      0.79      0.64        24
                  CHEF       0.85      0.71      0.77        24
          CONSTRUCTION       0.78      0.82      0.80        22
            CONSULTANT       0.67      0.26      0.38        23
              DESIGNER       0.86      0.86      0.86        21
         DIGITAL-MEDIA       0.69      

N-gram and vocabulary-pruning experiments (both eventually superseded by hyperparameter tuning below, kept as documented negative/neutral results)

In [36]:
vectorizer_bi = TfidfVectorizer(ngram_range=(1, 2))
x_train_vec_bi = vectorizer_bi.fit_transform(x_train_str)
x_test_vec_bi = vectorizer_bi.transform(x_test_str)

model_bi = LogisticRegression(class_weight='balanced')
model_bi.fit(x_train_vec_bi, y_train)
y_pred_bi = model_bi.predict(x_test_vec_bi)
print(classification_report(y_test, y_pred_bi, zero_division=0))


                        precision    recall  f1-score   support

            ACCOUNTANT       0.51      0.83      0.63        24
              ADVOCATE       0.39      0.46      0.42        24
           AGRICULTURE       0.78      0.54      0.64        13
               APPAREL       0.50      0.16      0.24        19
                  ARTS       0.62      0.24      0.34        21
            AUTOMOBILE       1.00      0.29      0.44         7
              AVIATION       0.85      0.74      0.79        23
               BANKING       0.88      0.65      0.75        23
                   BPO       0.00      0.00      0.00         4
  BUSINESS-DEVELOPMENT       0.48      0.88      0.62        24
                  CHEF       0.81      0.71      0.76        24
          CONSTRUCTION       0.82      0.82      0.82        22
            CONSULTANT       1.00      0.13      0.23        23
              DESIGNER       0.82      0.86      0.84        21
         DIGITAL-MEDIA       0.79      

In [37]:
vectorizer_pruned = TfidfVectorizer(min_df=2, max_df=0.85)
x_train_vec_pruned = vectorizer_pruned.fit_transform(x_train_str)
x_test_vec_pruned = vectorizer_pruned.transform(x_test_str)

model_pruned = LogisticRegression(class_weight='balanced')
model_pruned.fit(x_train_vec_pruned, y_train)
y_pred_pruned = model_pruned.predict(x_test_vec_pruned)
print(classification_report(y_test, y_pred_pruned, zero_division=0))



                        precision    recall  f1-score   support

            ACCOUNTANT       0.57      0.83      0.68        24
              ADVOCATE       0.42      0.42      0.42        24
           AGRICULTURE       0.70      0.54      0.61        13
               APPAREL       0.67      0.21      0.32        19
                  ARTS       0.46      0.29      0.35        21
            AUTOMOBILE       0.57      0.57      0.57         7
              AVIATION       0.86      0.78      0.82        23
               BANKING       0.94      0.65      0.77        23
                   BPO       0.67      0.50      0.57         4
  BUSINESS-DEVELOPMENT       0.51      0.79      0.62        24
                  CHEF       0.81      0.71      0.76        24
          CONSTRUCTION       0.82      0.82      0.82        22
            CONSULTANT       0.38      0.13      0.19        23
              DESIGNER       0.81      0.81      0.81        21
         DIGITAL-MEDIA       0.79      

In [38]:
len(vectorizer.get_feature_names_out())

31856

In [39]:
len(vectorizer_pruned.get_feature_names_out())

15006

Hyperparameter tuning (GridSearchCV + cross-validation) - final model selection

In [40]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000))
])

param_grid = {
    'tfidf__min_df': [1, 2, 3],
    'tfidf__max_df': [0.85, 0.9, 1.0],
    'clf__C': [0.01, 0.1, 1, 10]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    pipeline, param_grid, cv=cv, scoring='f1_macro', n_jobs=-1
)
grid_search.fit(x_train_str, y_train)

print("Best params:", grid_search.best_params_)
print("Best CV macro-F1:", grid_search.best_score_)


Best params: {'clf__C': 10, 'tfidf__max_df': 0.85, 'tfidf__min_df': 2}
Best CV macro-F1: 0.6287575864869215


In [41]:
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(x_test_str)
print(classification_report(y_test, y_pred_best, zero_division=0))


                        precision    recall  f1-score   support

            ACCOUNTANT       0.59      0.83      0.69        24
              ADVOCATE       0.52      0.62      0.57        24
           AGRICULTURE       0.78      0.54      0.64        13
               APPAREL       0.50      0.37      0.42        19
                  ARTS       0.47      0.43      0.45        21
            AUTOMOBILE       0.75      0.43      0.55         7
              AVIATION       0.85      0.74      0.79        23
               BANKING       0.84      0.70      0.76        23
                   BPO       0.50      0.25      0.33         4
  BUSINESS-DEVELOPMENT       0.54      0.79      0.64        24
                  CHEF       0.84      0.67      0.74        24
          CONSTRUCTION       0.78      0.82      0.80        22
            CONSULTANT       0.60      0.26      0.36        23
              DESIGNER       0.84      0.76      0.80        21
         DIGITAL-MEDIA       0.67      

In [42]:
pipeline_rf = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', RandomForestClassifier(random_state=42))
])

param_grid_rf = {
    'tfidf__min_df': [1, 2],
    'tfidf__max_df': [0.85, 1.0],
    'clf__n_estimators': [200, 400,600,1000],
    'clf__max_depth': [None, 30],
    'clf__class_weight': ['balanced', 'balanced_subsample']
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search_rf = GridSearchCV(
    pipeline_rf, param_grid_rf, cv=cv, scoring='f1_macro', n_jobs=-1
)
grid_search_rf.fit(x_train_str, y_train)

print("Best params:", grid_search_rf.best_params_)
print("Best CV macro-F1:", grid_search_rf.best_score_)

best_rf = grid_search_rf.best_estimator_
y_pred_rf = best_rf.predict(x_test_str)
print(classification_report(y_test, y_pred_rf, zero_division=0))


Best params: {'clf__class_weight': 'balanced', 'clf__max_depth': 30, 'clf__n_estimators': 600, 'tfidf__max_df': 0.85, 'tfidf__min_df': 2}
Best CV macro-F1: 0.6099781732646521
                        precision    recall  f1-score   support

            ACCOUNTANT       0.51      0.88      0.65        24
              ADVOCATE       0.88      0.58      0.70        24
           AGRICULTURE       1.00      0.54      0.70        13
               APPAREL       0.75      0.32      0.44        19
                  ARTS       1.00      0.05      0.09        21
            AUTOMOBILE       1.00      0.29      0.44         7
              AVIATION       0.78      0.78      0.78        23
               BANKING       0.78      0.61      0.68        23
                   BPO       1.00      0.25      0.40         4
  BUSINESS-DEVELOPMENT       0.50      0.75      0.60        24
                  CHEF       0.83      0.83      0.83        24
          CONSTRUCTION       0.76      0.86      0.81   

In [37]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold

pipeline_svm = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LinearSVC(class_weight='balanced', max_iter=5000))
])

param_grid_svm = {
    'tfidf__min_df': [1, 2, 3],
    'tfidf__max_df': [0.85, 0.9, 1.0],
    'clf__C': [0.01, 0.1, 1, 10, 100],
    'clf__loss': ['hinge', 'squared_hinge'],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search_svm = GridSearchCV(
    pipeline_svm, param_grid_svm, cv=cv, scoring='f1_macro', n_jobs=-1
)
grid_search_svm.fit(x_train_str, y_train)

print("Best params:", grid_search_svm.best_params_)
print("Best CV macro-F1:", grid_search_svm.best_score_)

best_svm = grid_search_svm.best_estimator_
y_pred_svm_best = best_svm.predict(x_test_str)
print(classification_report(y_test, y_pred_svm_best, zero_division=0))


Best params: {'clf__C': 1, 'clf__loss': 'squared_hinge', 'tfidf__max_df': 1.0, 'tfidf__min_df': 2}
Best CV macro-F1: 0.6379716422879048
                        precision    recall  f1-score   support

            ACCOUNTANT       0.57      0.83      0.68        24
              ADVOCATE       0.67      0.67      0.67        24
           AGRICULTURE       0.89      0.62      0.73        13
               APPAREL       0.54      0.37      0.44        19
                  ARTS       0.50      0.38      0.43        21
            AUTOMOBILE       0.75      0.43      0.55         7
              AVIATION       0.86      0.78      0.82        23
               BANKING       0.76      0.70      0.73        23
                   BPO       0.50      0.25      0.33         4
  BUSINESS-DEVELOPMENT       0.55      0.75      0.63        24
                  CHEF       0.85      0.71      0.77        24
          CONSTRUCTION       0.78      0.82      0.80        22
            CONSULTANT       0.

### Model comparison (after tuning)

| Model | Best CV macro-F1 | Test accuracy | Test macro-F1 |
|---|---|---|---|
| Logistic Regression | 0.629 | 0.67 | 0.65 |
| Random Forest | 0.610 | 0.67 | 0.63 |
| **Linear SVM (winner)** | **0.638** | **0.69** | **0.67** |

Linear SVM comes out ahead on both cross-validated and held-out macro-F1,
so it's the model shipped in `main.py` / `model/resume_classifier.joblib`.

In [ ]:
import joblib
joblib.dump(grid_search_svm.best_estimator_, '/kaggle/working/resume_classifier.joblib')

Step 4: PDF ingestion

In [44]:
import pdfplumber 

In [45]:
def extract_text_from_pdf(pdf_path:str)->str:
    with pdfplumber.open(pdf_path) as pdf:
        text=[]
        for i in range(len(pdf.pages)):
            text.append(pdf.pages[i].extract_text() or '')
    return '\n'.join(text)
    

In [46]:
test_sample=extract_text_from_pdf("dataset/archive (5)/data/data/AVIATION/10189110.pdf")

In [47]:
cleaned = clean_text(test_sample)
tokens = preprocessing(cleaned)
tokens_str = ' '.join(tokens)
grid_search.best_estimator_.predict([tokens_str])


array(['AVIATION'], dtype=object)

Step 5: Entity extraction (NER) - hybrid: spaCy for DATE, HuggingFace transformer (scored) for PERSON/ORG/LOCATION

In [48]:
def extract_entities(text: str) -> dict:
    doc = nlp(text, disable=["parser"])
    entities = {"PERSON": [], "ORG": [], "GPE": [], "DATE": []}
    for ent in doc.ents:
        if ent.label_ in entities:
            entities[ent.label_].append(ent.text)
    return entities


In [49]:
df['Resume_clean'][0]

"HR ADMINISTRATOR/MARKETING ASSOCIATE\nHR ADMINISTRATOR Summary Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management. Respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to customer service. Highlights Focused on customer satisfaction Team management Marketing savvy Conflict resolution techniques Training and development Skilled multi-tasker Client relations specialist Accomplishments Missouri DOT Supervisor Training Certification Certified by IHG in Customer Loyalty and Marketing by Segment Hilton Worldwide General Manager Training Certification Accomplished Trainer for cross server hospitality systems such as Hilton OnQ , Micros Opera PMS , Fidelio OPERA Reservation System (ORS) , Holidex Completed courses and seminars in customer service, sales strategies, inventory control, loss prevention, safety, time management, leadership and performance assessment. Experience HR A

In [51]:
from transformers import pipeline as hf_pipeline

ner_pipeline = hf_pipeline(
    "ner", model="dslim/bert-base-NER", aggregation_strategy="simple", device=0
)




In [52]:
def extract_entities_scored(text: str, threshold: float = 0.8) -> dict:
    results = ner_pipeline(text, stride=50)
    entities = {"PER": [], "ORG": [], "LOC": [], "MISC": []}
    for ent in results:
        if ent["score"] >= threshold and ent["entity_group"] in entities:
            entities[ent["entity_group"]].append((ent["word"], round(float(ent["score"]), 3)))
    return entities


In [53]:
extract_entities_scored(df['Resume_clean'][0].strip(),0.80)

{'PER': [],
 'ORG': [('Hospitality and Customer Service Management', 0.982),
  ('IHG', 0.807),
  ('Hilton Worldwide', 0.907),
  ('Hilton OnQ', 0.968),
  ('Micros Opera PM', 0.968),
  ('Fidelio OPERA Reservation System', 0.953),
  ('ORS', 0.91),
  ('Holidex', 0.861),
  ('Chamber of Commerce', 0.997),
  ('Jefferson College', 0.922),
  ('State Business Administration', 0.989),
  ('State', 0.853)],
 'LOC': [],
 'MISC': [('I', 0.867)]}

Step 6: Skills & Education extraction (PhraseMatcher)

In [122]:
# --- Inlined from skills_keywords.py for Kaggle portability ---
# (kept as a separate importable module in the actual project;
# see Resume ATS system/skills_keywords.py for the source of truth)

"""
Curated keyword lists for gazetteer-based extraction (via spaCy's PhraseMatcher)
of skills and education level, since general-purpose NER (spaCy's built-in
entities, or a CoNLL-trained transformer) has no SKILL/EDUCATION category and
will misclassify or ignore these terms - confirmed empirically on real resumes,
e.g. "Machine Learning", "CNN", "Keras" being mistagged as ORG.

Organized by the domains present in the 24-category resume dataset, since a
tech-only list would be useless for e.g. CHEF or AVIATION resumes. Each list
is deliberately flat (not nested) so it can be fed directly into a PhraseMatcher.

NOTE on AMBIGUOUS_SKILLS: short abbreviations like "R", "CV", "AI" are prone to
false positives (e.g. "CV" meaning "curriculum vitae" on a resume, not Computer
Vision). This list only flags which terms are risky - the matcher built on top
of this file needs to handle them with case-sensitive/standalone-token matching
rather than the case-insensitive matching used for everything else.
"""

TECH_DATA_SKILLS = [
    # Programming languages
    "Python", "Java", "JavaScript", "TypeScript", "C", "C++", "C#",
    "R", "SQL", "PHP", "Ruby", "Go", "Golang", "Kotlin", "Swift",
    "Scala", "MATLAB", "Bash", "Shell Scripting", "PowerShell",

    # AI and machine learning
    "Artificial Intelligence", "AI", "Machine Learning", "ML",
    "Deep Learning", "DL", "Computer Vision", "CV",
    "Natural Language Processing", "NLP",
    "Generative AI", "Large Language Models", "LLM", "LLMs",
    "Prompt Engineering", "Transformers", "BERT", "GPT",
    "Retrieval-Augmented Generation", "RAG",
    "Recommendation Systems", "Predictive Modeling",
    "Supervised Learning", "Unsupervised Learning",
    "Reinforcement Learning", "Transfer Learning",
    "Feature Engineering", "Model Deployment",
    "Model Evaluation", "Hyperparameter Tuning",
    "Time Series Analysis", "Anomaly Detection",

    # Neural network concepts
    "Neural Network", "Artificial Neural Network", "ANN",
    "Convolutional Neural Network", "CNN",
    "Recurrent Neural Network", "RNN",
    "Long Short-Term Memory", "LSTM",
    "Generative Adversarial Network", "GAN",
    "EfficientNet", "ResNet", "YOLO", "Vision Transformer", "ViT",

    # AI and data libraries
    "TensorFlow", "PyTorch", "Keras", "Scikit-learn",
    "OpenCV", "Pandas", "NumPy", "SciPy", "Matplotlib",
    "Seaborn", "Hugging Face", "XGBoost", "LightGBM", "CatBoost", "Ultralytics",

    # Data science and analytics
    "Data Analysis", "Data Analytics", "Data Science",
    "Statistical Analysis", "Statistics", "Data Mining",
    "Data Visualization", "Exploratory Data Analysis", "EDA",
    "Business Intelligence", "BI", "A/B Testing", "Experiment Design",

    # Databases
    "MySQL", "PostgreSQL", "Postgres", "SQL Server",
    "Microsoft SQL Server", "Oracle Database", "SQLite",
    "MongoDB", "Redis", "Cassandra", "DynamoDB",
    "Elasticsearch", "Neo4j", "Firebase", "Supabase",
    "Vector Database", "Pinecone", "FAISS", "ChromaDB", "pgvector",

    # Data engineering
    "ETL", "ELT", "Data Pipeline", "Data Warehousing", "Data Warehouse",
    "Data Lake", "Big Data", "Apache Spark", "Spark", "Hadoop",
    "Apache Kafka", "Kafka", "Airflow", "Apache Airflow",
    "Databricks", "Snowflake", "dbt",

    # Backend and APIs
    "REST API", "RESTful API", "API Development",
    "FastAPI", "Django", "Flask", "Spring Boot",
    "Node.js", "Express.js", "GraphQL", "Microservices",

    # Frontend and mobile
    "React", "Angular", "Vue.js", "HTML", "HTML5", "CSS", "CSS3",
    "Bootstrap", "Flutter", "React Native", "Android Development", "iOS Development",

    # Cloud and DevOps
    "AWS", "Amazon Web Services", "Azure", "Microsoft Azure",
    "Google Cloud", "Google Cloud Platform", "GCP",
    "Docker", "Kubernetes", "Terraform",
    "Jenkins", "GitHub Actions", "CI/CD",
    "DevOps", "MLOps", "MLflow", "Kubeflow", "Linux", "Ubuntu",

    # Version control and development tools
    "Git", "GitHub", "GitLab", "Bitbucket",
    "Jupyter Notebook", "Google Colab", "Visual Studio Code",

    # IT support
    "IT Support", "Technical Support", "Help Desk",
    "Hardware Troubleshooting", "Software Troubleshooting",
    "Network Troubleshooting", "Active Directory",
    "Windows Server", "TCP/IP", "DNS", "DHCP", "VPN",
]

BUSINESS_FINANCE_SKILLS = [
    # Finance and accounting
    "Financial Analysis", "Financial Reporting", "Financial Planning",
    "Financial Modeling", "Budgeting", "Forecasting", "Accounting",
    "General Ledger", "Bookkeeping", "QuickBooks",
    "Accounts Payable", "Accounts Receivable",
    "Payroll", "Auditing", "Internal Audit", "External Audit",
    "Tax Preparation", "Tax Accounting", "Bank Reconciliation",
    "Balance Sheet", "Income Statement", "Cash Flow",
    "Cost Accounting", "Management Accounting", "IFRS", "GAAP",
    "Risk Management", "Credit Analysis", "Investment Analysis",
    "Portfolio Management", "Treasury Management",

    # Sales and business development
    "Sales", "Sales Strategy", "B2B Sales", "B2C Sales",
    "Inside Sales", "Outside Sales", "Retail Sales",
    "Business Development", "Lead Generation", "Prospecting",
    "Cold Calling", "Pipeline Management", "Sales Forecasting",
    "Account Management", "Key Account Management",
    "Client Relationship Management", "CRM", "Salesforce", "HubSpot",
    "Negotiation", "Contract Negotiation", "Closing",
    "Upselling", "Cross-selling",

    # Operations and procurement
    "Operations Management", "Business Operations",
    "Process Improvement", "Process Optimization", "Cost Reduction",
    "Vendor Management", "Supplier Management", "Procurement",
    "Purchasing", "Sourcing", "Strategic Sourcing",
    "Contract Management", "Inventory Management", "Supply Chain Management",

    # Marketing
    "Marketing", "Digital Marketing", "Market Research", "Marketing Strategy",
    "Brand Management", "Product Marketing", "Email Marketing",
    "Content Marketing", "Search Engine Optimization", "SEO",
    "Search Engine Marketing", "SEM", "Google Analytics", "Google Ads",
    "Meta Ads", "Facebook Ads", "Campaign Management",

    # Administration
    "Office Administration", "Administrative Support", "Executive Assistance",
    "Calendar Management", "Meeting Coordination", "Travel Coordination",
    "Record Keeping", "Document Management", "Data Entry", "Report Preparation",
    "Microsoft Office", "Microsoft Excel", "Excel",
    "Microsoft Word", "Microsoft PowerPoint",
]

HR_SKILLS = [
    "Recruitment", "Onboarding", "Employee Relations", "Benefits Administration",
    "Performance Management", "Talent Acquisition", "HRIS", "Compensation",
    "Labor Relations", "Training and Development",
    "Workforce Planning", "Employee Engagement", "Compliance",
]

HEALTHCARE_SKILLS = [
    "Patient Care", "Clinical Research", "Nursing", "CPR", "HIPAA",
    "Medical Billing", "Medical Coding", "Electronic Health Records", "EHR",
    "Phlebotomy", "Vital Signs", "Patient Assessment", "Pharmacology",
    "ICD-9", "ICD-10", "CPT", "Infection Control",
]

CONSTRUCTION_ENGINEERING_SKILLS = [
    "AutoCAD", "Blueprint Reading", "Project Management", "OSHA",
    "Structural Engineering", "Civil Engineering", "Solidworks",
    "Quality Control", "Scheduling", "Cost Estimation", "Site Safety",
    "Welding", "Electrical Systems", "HVAC", "Plumbing", "Carpentry",
]

CULINARY_SKILLS = [
    "Menu Planning", "Food Safety", "ServSafe", "Culinary Arts",
    "Inventory Management", "Food Preparation", "Kitchen Management",
    "Catering", "Food Cost Control", "Sanitation", "Baking", "Pastry",
]

AVIATION_LOGISTICS_SKILLS = [
    "FAA Regulations", "Aircraft Maintenance", "Logistics", "Flight Operations",
    "Supply Chain", "Inventory Control", "Shipping", "Warehouse Management",
    "AOG", "Purchasing", "Quality Assurance", "Expediting",
]

DESIGN_ARTS_MEDIA_SKILLS = [
    "Adobe Photoshop", "Adobe Illustrator", "Adobe InDesign", "Graphic Design",
    "UI/UX", "Figma", "Content Creation", "Social Media Marketing",
    "Video Editing", "Copywriting", "Branding", "Typography", "Web Design",
]

BPO_CUSTOMER_SUPPORT_SKILLS = [
    # Core customer support
    "Customer Service", "Customer Support", "Customer Care", "Client Support",
    "Technical Support", "Help Desk", "IT Help Desk", "Service Desk",
    "Call Center", "Contact Center", "BPO", "Business Process Outsourcing",

    # Communication channels
    "Inbound Calls", "Outbound Calls", "Call Handling", "Phone Support",
    "Voice Support", "Non-Voice Support", "Email Support", "Chat Support",
    "Live Chat Support", "Omnichannel Support", "Social Media Support",

    # Customer interaction
    "Complaint Resolution", "Customer Retention", "Customer Escalation",
    "Escalation Management", "De-escalation", "Active Listening", "Empathy",
    "Rapport Building", "Upselling", "Cross-selling", "Lead Generation",
    "Appointment Setting", "Telemarketing",

    # Call-center metrics
    "AHT", "Average Handle Time", "Average Handling Time",
    "FCR", "First Call Resolution",
    "CSAT", "Customer Satisfaction", "Customer Satisfaction Score",
    "NPS", "Net Promoter Score", "SLA", "Service Level Agreement",
    "Quality Assurance", "Quality Monitoring", "Call Monitoring",
    "Call Auditing", "KPI", "Key Performance Indicators",
    "Workforce Management", "WFM",

    # Tools
    "Zendesk", "Freshdesk", "ServiceNow", "Salesforce Service Cloud",
    "HubSpot", "Intercom", "Five9", "Genesys", "Avaya", "Cisco Finesse",
    "Zoho CRM", "Microsoft Dynamics 365", "Ticketing System", "CRM Software",

    # Operational skills
    "Ticket Management", "Case Management", "Order Processing",
    "Refund Processing", "Account Verification", "Data Entry",
    "Documentation", "Knowledge Base", "Troubleshooting",
    "Remote Support", "Back Office Support",
]

EDUCATION_TEACHING_SKILLS = [
    "Teaching", "Classroom Management", "Lesson Planning",
    "Curriculum Development", "Curriculum Design",
    "Student Assessment", "Educational Technology",
    "Instructional Design", "Tutoring", "Mentoring",
    "Special Education", "Early Childhood Education",
    "E-learning", "Learning Management System", "LMS",
    "Blackboard", "Moodle", "Google Classroom",
]

LEGAL_SKILLS = [
    "Legal Research", "Legal Writing", "Litigation",
    "Contract Drafting", "Contract Review", "Case Management",
    "Legal Compliance", "Corporate Law", "Commercial Law",
    "Civil Law", "Criminal Law", "Intellectual Property",
    "Due Diligence", "Document Review", "Paralegal", "Westlaw", "LexisNexis",
]

SECURITY_SKILLS = [
    "Security Operations", "Physical Security", "Access Control",
    "Surveillance", "CCTV", "Incident Reporting", "Risk Assessment",
    "Emergency Response", "Loss Prevention", "Cybersecurity",
    "Information Security", "Network Security", "Penetration Testing",
    "Vulnerability Assessment", "SIEM", "SOC", "Security Operations Center",
    "Firewalls", "Incident Response",
]

AUTOMOTIVE_MECHANICAL_SKILLS = [
    "Automotive Repair", "Vehicle Maintenance", "Preventive Maintenance",
    "Mechanical Engineering", "Mechanical Maintenance", "Diagnostics",
    "Engine Repair", "Brake Repair", "Transmission Repair",
    "Hydraulics", "Pneumatics", "CNC", "Machining",
    "Technical Drawing", "SolidWorks", "CATIA",
]

FITNESS_SPORTS_SKILLS = [
    "Personal Training", "Fitness Training", "Strength Training",
    "Cardiovascular Training", "Exercise Programming", "Sports Coaching",
    "Nutrition Coaching", "Group Fitness", "First Aid",
    "Injury Prevention", "Body Composition Assessment",
]

RETAIL_HOSPITALITY_SKILLS = [
    "Retail Operations", "Point of Sale", "POS", "Cash Handling",
    "Merchandising", "Visual Merchandising", "Store Management",
    "Hospitality", "Hotel Operations", "Front Desk", "Guest Relations",
    "Reservations", "Housekeeping", "Food and Beverage", "Event Planning",
]

MANUFACTURING_SKILLS = [
    "Manufacturing", "Production Planning", "Production Management",
    "Lean Manufacturing", "Six Sigma", "Kaizen", "5S",
    "Quality Management", "Quality Control", "Root Cause Analysis",
    "CAPA", "GMP", "ISO 9001", "Assembly Line",
]

SOFT_SKILLS = [
    "Leadership", "Communication", "Teamwork", "Problem Solving",
    "Time Management", "Critical Thinking", "Adaptability",
    "Multitasking", "Attention to Detail",
    "Public Speaking", "Conflict Resolution", "Decision Making",
]

EDUCATION_LEVELS = [
    # Doctoral
    "PhD", "Ph.D.", "Doctorate", "Doctoral Degree", "Doctor of Philosophy",
    "DBA", "Doctor of Business Administration",
    "EdD", "Ed.D.", "Doctor of Education",
    "MD", "M.D.", "Doctor of Medicine", "Juris Doctor", "JD", "J.D.",

    # Master's
    "Master's Degree", "Master of Science", "MSc", "M.S.",
    "Master of Arts", "M.A.",
    "Master of Engineering", "MEng", "M.Eng.",
    "Master of Technology", "MTech", "M.Tech.",
    "MBA", "Master of Business Administration",
    "Postgraduate Degree", "Postgraduate Diploma", "PGDip",

    # Bachelor's
    "Bachelor's Degree", "Bachelor of Science", "BSc", "B.S.",
    "Bachelor of Arts", "B.A.",
    "Bachelor of Engineering", "BEng", "B.Eng.",
    "Bachelor of Technology", "BTech", "B.Tech.",
    "Bachelor of Business Administration", "BBA", "B.B.A.",
    "Bachelor of Commerce", "BCom", "B.Com.", "Undergraduate Degree",

    # Associate
    "Associate Degree", "Associate's Degree",
    "Associate of Science", "A.S.", "Associate of Arts", "A.A.",
    "Associate of Applied Science", "AAS", "A.A.S.",

    # Secondary education
    "High School Diploma", "Secondary School Certificate",
    "Secondary Education", "GED", "General Educational Development",
    "A Levels", "A-Levels", "GCSE", "IGCSE",

    # Vocational
    "Diploma", "Technical Diploma", "Vocational Diploma",
    "Professional Diploma", "Certificate", "Certification",
    "Professional Certificate", "Technical Certificate",
    "Trade School", "Vocational Training",
]

# Short abbreviations that risk false positives with plain case-insensitive
# matching (e.g. lowercase "cv" meaning "curriculum vitae", not Computer
# Vision). The matcher built on top of this file should handle these with
# case-sensitive/standalone-token matching, not the default case-insensitive
# pass used for everything else.
AMBIGUOUS_SKILLS = {"R", "C", "AI", "CV", "ML", "DL", "BI"}

# Maps common variants/abbreviations to one canonical form, so e.g. "GCP",
# "Google Cloud", and "Google Cloud Platform" aren't treated as three
# unrelated skills in downstream output.
SKILL_ALIASES = {
    "gcp": "Google Cloud Platform",
    "google cloud": "Google Cloud Platform",
    "google cloud platform": "Google Cloud Platform",
    "aws": "Amazon Web Services",
    "amazon web services": "Amazon Web Services",
    "ai": "Artificial Intelligence",
    "ml": "Machine Learning",
    "machine learning": "Machine Learning",
    "dl": "Deep Learning",
    "deep learning": "Deep Learning",
    "cv": "Computer Vision",
    "computer vision": "Computer Vision",
    "nlp": "Natural Language Processing",
    "natural language processing": "Natural Language Processing",
    "bi": "Business Intelligence",
    "business intelligence": "Business Intelligence",
    "cnn": "Convolutional Neural Network",
    "convolutional neural network": "Convolutional Neural Network",
    "rnn": "Recurrent Neural Network",
    "recurrent neural network": "Recurrent Neural Network",
    "lstm": "Long Short-Term Memory",
    "long short-term memory": "Long Short-Term Memory",
    "gan": "Generative Adversarial Network",
    "generative adversarial network": "Generative Adversarial Network",
    "llm": "Large Language Models",
    "llms": "Large Language Models",
    "large language models": "Large Language Models",
    "rag": "Retrieval-Augmented Generation",
    "retrieval-augmented generation": "Retrieval-Augmented Generation",
    "vit": "Vision Transformer",
    "vision transformer": "Vision Transformer",
    "eda": "Exploratory Data Analysis",
    "exploratory data analysis": "Exploratory Data Analysis",
    "sklearn": "Scikit-learn",
    "scikit learn": "Scikit-learn",
    "scikit-learn": "Scikit-learn",
    "nodejs": "Node.js",
    "node.js": "Node.js",
    "postgres": "PostgreSQL",
    "postgresql": "PostgreSQL",
    "ms excel": "Microsoft Excel",
    "microsoft excel": "Microsoft Excel",
    "excel": "Microsoft Excel",
    "customer care": "Customer Service",
    "customer support": "Customer Service",
    "customer service": "Customer Service",
    "average handle time": "Average Handling Time",
    "average handling time": "Average Handling Time",
    "aht": "Average Handling Time",
    "first call resolution": "First Call Resolution",
    "fcr": "First Call Resolution",
    "customer satisfaction score": "Customer Satisfaction Score",
    "customer satisfaction": "Customer Satisfaction Score",
    "csat": "Customer Satisfaction Score",
}


def canonicalize_skill(skill: str) -> str:
    """
    Map a matched skill to its canonical form.

    Checks SKILL_ALIASES first (for abbreviations/variants that should
    collapse to a different canonical name), then falls back to the
    properly-cased version already defined in ALL_SKILLS (so a skill matched
    in whatever casing the resume happened to use, e.g. "feature engineering",
    still comes back as "Feature Engineering" instead of passing through
    unchanged). _CANONICAL_CASING is built after ALL_SKILLS at the bottom of
    this module; referencing it here is fine since it only needs to exist by
    call time, not by the time this function is defined.
    """
    key = skill.strip().lower()
    if key in SKILL_ALIASES:
        return SKILL_ALIASES[key]
    if key in _CANONICAL_CASING:
        return _CANONICAL_CASING[key]
    return skill.strip()


def deduplicate_phrases(phrases: list[str]) -> list[str]:
    """Remove exact duplicates (case-insensitively) while preserving order."""
    seen = set()
    result = []
    for phrase in phrases:
        cleaned = phrase.strip()
        normalized = cleaned.casefold()
        if cleaned and normalized not in seen:
            seen.add(normalized)
            result.append(cleaned)
    return result


SKILL_GROUPS = {
    "technical": TECH_DATA_SKILLS,
    "business_finance": BUSINESS_FINANCE_SKILLS,
    "human_resources": HR_SKILLS,
    "healthcare": HEALTHCARE_SKILLS,
    "construction_engineering": CONSTRUCTION_ENGINEERING_SKILLS,
    "culinary": CULINARY_SKILLS,
    "aviation_logistics": AVIATION_LOGISTICS_SKILLS,
    "design_media": DESIGN_ARTS_MEDIA_SKILLS,
    "bpo_customer_support": BPO_CUSTOMER_SUPPORT_SKILLS,
    "education_teaching": EDUCATION_TEACHING_SKILLS,
    "legal": LEGAL_SKILLS,
    "security": SECURITY_SKILLS,
    "automotive_mechanical": AUTOMOTIVE_MECHANICAL_SKILLS,
    "fitness_sports": FITNESS_SPORTS_SKILLS,
    "retail_hospitality": RETAIL_HOSPITALITY_SKILLS,
    "manufacturing": MANUFACTURING_SKILLS,
    "soft_skills": SOFT_SKILLS,
}

ALL_SKILLS = deduplicate_phrases(
    [skill for group in SKILL_GROUPS.values() for skill in group]
)

# Case-insensitive lookup used by canonicalize_skill() to restore proper
# casing for skills that don't have an explicit entry in SKILL_ALIASES.
_CANONICAL_CASING = {s.lower(): s for s in ALL_SKILLS}


# --- PhraseMatcher construction (was bundled in the original import cell -
# restored here after inlining the keyword lists above) ---
from spacy.matcher import PhraseMatcher

non_ambiguous = [s for s in ALL_SKILLS if s not in AMBIGUOUS_SKILLS]
skill_matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
skill_matcher.add("SKILL", [nlp.make_doc(s) for s in non_ambiguous])

ambiguous_matcher = PhraseMatcher(nlp.vocab)
ambiguous_matcher.add("AMBIGUOUS_SKILL", [nlp.make_doc(s) for s in AMBIGUOUS_SKILLS])

edu_matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
edu_matcher.add("EDUCATION", [nlp.make_doc(e) for e in EDUCATION_LEVELS])


In [123]:
def collect_matches(matcher, doc):
    results = []
    for match_id, start, end in matcher(doc):
        span = doc[start:end]
        results.append({"text": span.text, "start_char": span.start_char, "end_char": span.end_char})
    return results


def remove_overlapping_matches(matches):
    matches = sorted(
        matches,
        key=lambda item: (item["start_char"], -(item["end_char"] - item["start_char"]))
    )
    selected = []
    for current in matches:
        overlaps = any(
            current["start_char"] < saved["end_char"] and current["end_char"] > saved["start_char"]
            for saved in selected
        )
        if not overlaps:
            selected.append(current)
    return selected


In [124]:
def extract_skills_and_education(text: str) -> dict:
    doc = nlp(text)

    raw_skill_matches = collect_matches(skill_matcher, doc) + collect_matches(ambiguous_matcher, doc)
    skill_matches = remove_overlapping_matches(raw_skill_matches)

    canonical_skills = []
    seen = set()
    for m in skill_matches:
        canonical = canonicalize_skill(m["text"])
        if canonical.lower() not in seen:
            seen.add(canonical.lower())
            canonical_skills.append(canonical)

    edu_matches = remove_overlapping_matches(collect_matches(edu_matcher, doc))
    education = []
    seen_edu = set()
    for m in edu_matches:
        if m["text"].lower() not in seen_edu:
            seen_edu.add(m["text"].lower())
            education.append(m["text"])

    return {"skills": canonical_skills, "education": education}


In [125]:
extract_skills_and_education(df['Resume_clean'][0].strip())

{'skills': ['Marketing',
  'Customer Service',
  'Hospitality',
  'Customer Satisfaction Score',
  'Conflict Resolution',
  'Training and Development',
  'Sales',
  'Inventory Control',
  'Loss Prevention',
  'Time Management',
  'Leadership',
  'Compensation',
  'Labor Relations',
  'Documentation',
  'Statistics',
  'Employee Relations',
  'ICD-9',
  'CPT',
  'Medical Billing',
  'Data Analysis',
  'Budgeting',
  'Accounting',
  'Payroll',
  'Purchasing',
  'Swift'],
 'education': ['Certification', 'High School Diploma']}

Step 7: Job matching

In [ ]:
vectorizer = grid_search_svm.best_estimator_.named_steps["tfidf"]

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

_SOFT_SKILLS_LOWER = {s.lower() for s in SOFT_SKILLS}


def _skills_for_matching(raw_text: str) -> tuple[str, set[str]]:
    """
    Extract skills for job/resume matching, reduced from raw text - strips
    out narrative language ("we are seeking...") and job-history verbs that
    don't reflect actual required competencies. Classification still uses
    the full preprocessed text; this reduction is specific to matching.

    Soft skills (Teamwork, Communication, Time Management, etc.) are
    excluded here specifically - verified this matters: a real Junior Chef
    posting's required-skills list was 6 soft skills out of 11 total, and
    an unrelated ML/AI resume outscored an actual chef's resume on
    skill_coverage (0.36 vs 0.091) purely because soft-skill buzzwords
    appear on almost any professional resume regardless of domain. Soft
    skills don't differentiate domain fit, so they're excluded from
    matching (they're still shown normally in extract_skills_and_education's
    general output - this exclusion is specific to job/resume matching).

    Returns both a joined string (for TF-IDF cosine similarity) and a
    lowercase set (for skill-coverage calculation), computed from a single
    extraction pass rather than extracting twice.
    """
    skills = extract_skills_and_education(clean_text(raw_text))["skills"]
    hard_skills = [s for s in skills if s.lower() not in _SOFT_SKILLS_LOWER]
    return ' '.join(hard_skills), {s.lower() for s in hard_skills}


def rank_resumes_for_job(job_description: str, resumes: dict[str, str]) -> list[dict]:
    """
    resumes: {identifier (e.g. filename): RAW resume text}

    Returns a list of dicts sorted by cosine_similarity descending, each with:
      - candidate: the identifier
      - cosine_similarity: overall similarity between extracted skill sets.
        Can be misleadingly low for a candidate with many skills beyond
        what the job needs - cosine similarity is diluted by "extra"
        content even when every required skill is present (verified: a
        candidate with only the 5 exact required skills scored 1.0, the
        same candidate's real 43-skill resume scored 0.389, despite having
        all 5 required skills - having more skills than needed actively
        lowers this score). skill_coverage below doesn't have that problem.
      - skill_coverage: fraction of the job's required skills the candidate
        actually has (0.0-1.0) - unaffected by how many extra skills the
        candidate also has.
    """
    job_text, job_skill_set = _skills_for_matching(job_description)
    job_vec = vectorizer.transform([job_text])

    names = list(resumes.keys())
    resume_texts = []
    resume_skill_sets = []
    for text in resumes.values():
        t, s = _skills_for_matching(text)
        resume_texts.append(t)
        resume_skill_sets.append(s)

    resume_vecs = vectorizer.transform(resume_texts)
    cosine_scores = cosine_similarity(job_vec, resume_vecs)[0]

    results = []
    for name, cos_score, resume_skills in zip(names, cosine_scores, resume_skill_sets):
        coverage = len(job_skill_set & resume_skills) / len(job_skill_set) if job_skill_set else 0.0
        results.append({
            "candidate": name,
            "cosine_similarity": float(cos_score),
            "skill_coverage": coverage,
        })

    return sorted(results, key=lambda r: r["cosine_similarity"], reverse=True)


In [131]:
description='''# Junior Chef

## Job Summary

We are looking for a reliable and motivated Junior Chef to support daily kitchen operations and help prepare high-quality meals.

The ideal candidate should have basic cooking experience, good knowledge of food safety, and the ability to work well in a busy kitchen environment.

## Responsibilities

* Prepare ingredients before cooking.
* Cook simple meals according to recipes.
* Assist senior chefs with food preparation.
* Maintain cleanliness in the kitchen.
* Follow food safety and sanitation rules.
* Check food quality before serving.
* Store ingredients correctly.
* Monitor basic inventory levels.
* Help receive and organize food supplies.
* Use kitchen tools and equipment safely.
* Follow portion sizes and presentation standards.
* Work with the kitchen team during busy shifts.

## Requirements

* Diploma or certificate in Culinary Arts is preferred.
* Zero to two years of kitchen experience.
* Basic knowledge of food preparation and cooking methods.
* Understanding of food safety and hygiene.
* Ability to work under pressure.
* Good teamwork and communication skills.
* Strong attention to detail.
* Ability to stand for long periods.
* Willingness to work flexible shifts, including weekends.

## Skills

* Food Preparation
* Cooking
* Kitchen Management
* Food Safety
* Sanitation
* Menu Preparation
* Inventory Management
* Knife Skills
* Time Management
* Teamwork
* Communication
* Attention to Detail
* Multitasking
* Problem Solving

## Preferred Qualifications

* Experience in restaurants, hotels, or catering.
* Knowledge of baking or pastry preparation.
* ServSafe certification or similar food safety training.
* Basic experience with food cost control.
'''

In [ ]:
x=1430
print(df['Category'][x])
rank_resumes_for_job(description, {df['ID'][x]: df['Resume_clean'][x]})


CHEF


[{'candidate': np.int64(29775391),
  'cosine_similarity': 0.6050027787759688,
  'skill_coverage': 0.6363636363636364}]

Step 8: Search a resume pool for the best candidate

In [ ]:
def find_best_candidates(job_description: str, resumes: dict[str, str], top_n: int = 5) -> list[dict]:
    """
    Search a pool of resumes and return the top_n best candidates for a job.

    Sorted by skill_coverage first, cosine_similarity as a tiebreaker - NOT
    by cosine_similarity alone. This matters: cosine_similarity can rank an
    over-qualified candidate (many skills beyond what's needed) below one
    with only the exact required skills, since "extra" content dilutes the
    similarity vector even when every requirement is met (verified earlier -
    a candidate with all 5 required skills plus 38 more scored 0.389, while
    a hypothetical candidate with only those exact 5 scored 1.0).
    skill_coverage answers "does this candidate meet the requirements,"
    which is what "best candidate" should actually mean here.
    """
    results = rank_resumes_for_job(job_description, resumes)
    results.sort(key=lambda r: (r["skill_coverage"], r["cosine_similarity"]), reverse=True)
    return results[:top_n]


In [ ]:
# Build a genuine competitive pool: a few resumes from mostly-irrelevant
# categories, plus a real CHEF resume as the one genuine fit for the
# Junior Chef posting above - a "search across one candidate" test
# wouldn't actually prove the ranking works. Uses only dataset-sourced
# resumes (no personal files), since this notebook is published on Kaggle.
pool = {}
for category, idx in [('AVIATION', 0), ('ARTS', 0), ('TEACHER', 0), ('HR', 0)]:
    row = df[df['Category'] == category].iloc[idx]
    pool[f"{category}_{row['ID']}"] = row['Resume_str']

chef_row = df[df['Category'] == 'CHEF'].iloc[0]
pool[f"CHEF_{chef_row['ID']}"] = chef_row['Resume_str']

best_candidates = find_best_candidates(description, pool, top_n=5)
best_candidates


[{'candidate': 'CHEF_15180322', 'cosine_similarity': 0.12151819917343112, 'skill_coverage': 0.09090909090909091}, {'candidate': 'AVIATION_69458502', 'cosine_similarity': 0.028054418649326868, 'skill_coverage': 0.0}, {'candidate': 'HR_16852973', 'cosine_similarity': 0.02796471971112044, 'skill_coverage': 0.0}, {'candidate': 'ARTS_31273413', 'cosine_similarity': 0.023299421487242185, 'skill_coverage': 0.0}, {'candidate': 'TEACHER_12467531', 'cosine_similarity': 0.01754031925753484, 'skill_coverage': 0.0}]


In [ ]:
def get_best_candidates(category: str, description: str, top_n: int = 5) -> list[dict]:
    """
    Search all resumes in df belonging to `category` for the best matches
    to a job description.

    Filters the category once, then makes a SINGLE efficient batched call
    to find_best_candidates - rather than one call per candidate, which
    would redundantly re-run clean_text + spaCy + PhraseMatcher on the
    (unchanged) job description once per resume in the category. For a
    category like CHEF (118 resumes), that's 118x wasted recomputation
    avoided by batching once instead.
    """
    subset = df[df['Category'] == category]
    pool = {row['ID']: row['Resume_clean'] for _, row in subset.iterrows()}
    return find_best_candidates(description, pool, top_n=top_n)


In [ ]:
# Real test: search the FULL CHEF category (118 resumes) against the
# Junior Chef posting above - proves the category search and the
# soft-skill fix both work correctly together across a large, realistic
# pool, not just a small hand-picked one.
get_best_candidates('CHEF', description, top_n=5)


[{'candidate': 27662298, 'cosine_similarity': 0.796790487358195, 'skill_coverage': 0.7272727272727273}, {'candidate': 29449419, 'cosine_similarity': 0.7053316022308203, 'skill_coverage': 0.6363636363636364}, {'candidate': 19268120, 'cosine_similarity': 0.6963046443441608, 'skill_coverage': 0.6363636363636364}, {'candidate': 53265899, 'cosine_similarity': 0.65184931661938, 'skill_coverage': 0.6363636363636364}, {'candidate': 20321582, 'cosine_similarity': 0.6303482287329166, 'skill_coverage': 0.6363636363636364}]
